# 04b · PyTorch lấy đạo hàm như thế nào?

**Chỉ học autograd, chưa train PINN.** Mục tiêu: hiểu `requires_grad=True`, `torch.autograd.grad`, `grad_outputs`, `create_graph=True`; kiểm tra đạo hàm bậc bốn bằng một đa thức biết đáp án. Mỗi đầu ra ở hàng $i$ chỉ phụ thuộc đầu vào cùng hàng.

## 1. Đạo hàm bậc một

PyTorch ghi lại các phép toán tạo $y=x^3$. Tọa độ phải có `requires_grad=True`. `grad_outputs=torch.ones_like(y)` có cùng shape với $y$ và cho PyTorch biết ta đang lấy tổng có trọng số 1 của các đầu ra; vì các hàng độc lập, kết quả là đạo hàm theo từng điểm.

In [ ]:
import torch

x = torch.tensor([[0.5], [1.0]], dtype=torch.float64, requires_grad=True)
y = x**3
dy = torch.autograd.grad(y, x, torch.ones_like(y), create_graph=True)[0]
print("autograd:", dy.reshape(-1).tolist())
print("đạo hàm tay 3*x**2:", (3*x**2).reshape(-1).tolist())

## 2. Vì sao `create_graph=True`?

Ta cần đạo hàm của **chính đạo hàm vừa tính**. Với $p(x)=x^4-2x^3+x$, kết quả mong đợi là $p^{(4)}(x)=24$. Viết từng bước để nhìn rõ bậc 1 → 2 → 3 → 4.

In [ ]:
x = torch.tensor([[0.25], [0.75]], dtype=torch.float64, requires_grad=True)
p = x**4 - 2*x**3 + x
p1 = torch.autograd.grad(p, x, torch.ones_like(p), create_graph=True)[0]
p2 = torch.autograd.grad(p1, x, torch.ones_like(p1), create_graph=True)[0]
p3 = torch.autograd.grad(p2, x, torch.ones_like(p2), create_graph=True)[0]
p4 = torch.autograd.grad(p3, x, torch.ones_like(p3), create_graph=True)[0]
print("p' tại hai điểm:", p1.reshape(-1).tolist())
print("p'''' tại hai điểm:", p4.reshape(-1).tolist())
assert torch.allclose(p4, torch.full_like(p4, 24.0))

## 3. Một hàm nhỏ để khỏi chép cùng câu lệnh

Bài PINN sẽ gọi autograd sáu lần trong mỗi bước train. Ta gói **một lệnh** vào hàm `grad(y,x)`; bản thân hàm không có phép toán mới.

In [ ]:
def grad(y, x):
    return torch.autograd.grad(y, x, torch.ones_like(y), create_graph=True)[0]

q1 = grad(p, x)
q2 = grad(q1, x)
q3 = grad(q2, x)
q4 = grad(q3, x)
print("hàm grad cho p'''' =", q4.reshape(-1).tolist())

## 4. Tách hai loại gradient

`grad(u,x)` là đạo hàm theo **tọa độ**, để lập residual phương trình. Sau này `loss.backward()` tính đạo hàm của loss theo **trọng số mạng**; `optimizer.step()` mới cập nhật trọng số. `grad` không tự train mạng.

**Tự làm:** trong một ô bản sao, bỏ `requires_grad=True` và ghi lỗi. Khôi phục ô gốc rồi thay $p(x)$ bằng $x^4$; đạo hàm bậc bốn thay đổi không? **Sản phẩm:** giá trị bậc một và bậc bốn kèm giải thích `create_graph=True`.

**Tiếp theo:** [04c · pipeline PINN](04c-pinn-pipeline.ipynb).

## 5. Đồ thị phép toán và quy tắc dây chuyền

Với z=2x+1, y=z³, PyTorch ghi đường đi x → z → y. Khi lấy dy/dx, nó đi ngược và nhân các đạo hàm cục bộ: dy/dx=3z²·2. grad_fn cho thấy tensor trung gian được tạo bởi phép toán có theo dõi.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
z = 2*x + 1
y = z**3
print(x.is_leaf, x.grad_fn)
print(z.grad_fn, y.grad_fn)
print(torch.autograd.grad(y, x)[0].item())  # 150


## 6. Vì sao phải bật requires_grad trước forward?

Nếu x không được theo dõi khi tính y, đường đi cần thiết để lấy đạo hàm theo x chưa được ghi. Bật cờ sau đó không sửa được y cũ; phải tính lại y.


In [ ]:
x_plain = torch.tensor(2.0)
y_plain = x_plain**3
print(x_plain.requires_grad, y_plain.grad_fn)  # False, None
x_plain.requires_grad_(True)
print(y_plain.grad_fn)  # y cũ không đổi
y_new = x_plain**3
print(torch.autograd.grad(y_new, x_plain)[0].item())


## 7. create_graph cho đạo hàm của đạo hàm

Nếu kết quả đạo hàm còn được lấy đạo hàm tiếp, cần create_graph=True. Nếu đạo hàm bậc một là kết quả cuối, có thể để False. Trong lúc train PINN dầm, kể cả đạo hàm bậc bốn cần True để loss còn truyền gradient về trọng số.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**3
dy = torch.autograd.grad(y, x, create_graph=True)[0]
d2y = torch.autograd.grad(dy, x)[0]
print(dy.item(), d2y.item())  # 12, 12


## 8. Khi nào tắt?

Chỉ dự đoán chuyển vị: dùng torch.no_grad và không cần tọa độ requires_grad. Nếu đánh giá mômen -u'', vẫn phải lấy đạo hàm theo tọa độ; đạo hàm đầu cần create_graph=True nhưng đạo hàm cuối có thể False. Chỉ detach sau khi tính xong, trước khi lưu/vẽ.


In [ ]:
from torch import nn
model = nn.Sequential(nn.Linear(1, 8), nn.Tanh(), nn.Linear(8, 1)).double()
x_test = torch.linspace(0, 1, 5, dtype=torch.float64).reshape(-1, 1)
with torch.no_grad():
    u_only = model(x_test)
print(u_only.shape, u_only.grad_fn)
x_m = x_test.clone().detach().requires_grad_(True)
u_m = model(x_m)
u_x = torch.autograd.grad(u_m, x_m, torch.ones_like(u_m), create_graph=True)[0]
u_xx = torch.autograd.grad(u_x, x_m, torch.ones_like(u_x), create_graph=False)[0]
moment = -u_xx.detach()
print(moment.shape, moment.grad_fn)


Tự làm: đổi create_graph=False ở lần tính u_x, rồi thử lấy u_xx; đặt forward của u_m trong no_grad và đọc lỗi. Giải thích hai lỗi bằng đồ thị phép toán.
